# Q1. 矩阵乘法与神经网络前向传播

一个两层神经网络，输入 $x \in \mathbb{R}^3$，隐藏层宽度为 4，输出为标量。

写出完整的前向传播公式（含激活函数 ReLU），并用纯 PyTorch（不调用 `nn.Linear`）手写实现。

$$h = \text{ReLU}(W_1 x + b_1), \quad W_1 \in \mathbb{R}^{4 \times 3}$$
$$y = W_2 h + b_2, \quad W_2 \in \mathbb{R}^{1 \times 4}$$

**验收标准**：输出 shape 正确，与 `nn.Linear` 的结果在数值上一致。

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ---------- 参数初始化 ----------
W1 = torch.randn(4, 3)
b1 = torch.randn(4)
W2 = torch.randn(1, 4)
b2 = torch.randn(1)

x = torch.randn(3)  # 输入向量

# ---------- 手写前向传播 ----------
h = torch.relu(W1 @ x + b1)   # shape: (4,)
y = W2 @ h + b2                # shape: (1,)

print("h shape:", h.shape)  # 期望 (4,)
print("y shape:", y.shape)  # 期望 (1,)
print("y value:", y)

h shape: torch.Size([4])
y shape: torch.Size([1])
y value: tensor([1.2640])


## 知识点

### 1. `nn.Linear` 不含激活函数

`nn.Linear` 只做纯线性变换：

$$\text{output} = x W^T + b$$

激活函数（ReLU、Sigmoid 等）需要**单独显式调用**。这是 PyTorch 的设计哲学——将线性层与非线性分离，组合更灵活。

```python
# 错误理解：以为 nn.Linear 自带激活
h = layer1(x)           # 这只是线性变换，没有 ReLU

# 正确写法：手动加激活
h = torch.relu(layer1(x))
```

---

### 2. 为什么写 `W @ x` 而不是 `x @ W^T`？

两者对 1D 向量**数学上等价**，但背后约定不同：

| 写法 | 惯例来源 | x 的 shape | 输出 shape |
|------|----------|-----------|-----------|
| `W @ x` | 数学（列向量）| `(3,)` | `(4,)` |
| `x @ W.T` | PyTorch / `nn.Linear` | `(3,)` 或 `(batch, 3)` | `(4,)` 或 `(batch, 4)` |

关键区别在于**批量输入**：

```python
x_batch = torch.randn(8, 3)   # batch_size=8

# W @ x_batch 会报错（shape 不兼容）
# x_batch @ W1.T 才能正确广播到 (8, 4)
h_batch = torch.relu(x_batch @ W1.T + b1)  # shape: (8, 4)
```

`nn.Linear` 内部正是用 `x @ W^T + b` 实现的，所以它天然支持任意 batch 维度。本题 x 无 batch 维，两种写法结果相同；实际工程中推荐用 `x @ W.T` 以保持一致性。

In [4]:
# ---------- 用 nn.Linear 验证数值一致性 ----------
layer1 = nn.Linear(3, 4)
layer2 = nn.Linear(4, 1)

# 将相同权重复制进 nn.Linear
with torch.no_grad():
    layer1.weight.copy_(W1)
    layer1.bias.copy_(b1)
    layer2.weight.copy_(W2)
    layer2.bias.copy_(b2)

h_ref = torch.relu(layer1(x))
y_ref = layer2(h_ref)

print("y_ref value:", y_ref)
print("数值一致:", torch.allclose(y, y_ref))

y_ref value: tensor([1.2640], grad_fn=<ViewBackward0>)
数值一致: True
